# Track 4: Aligned Response Refinement with Direct Preference Optimization (DPO)

This notebook demonstrates how to align a model with preferred human feedback using Direct Preference Optimization (DPO) and `DPOTrainer` from Hugging Face `trl`. We use **Unsloth** for memory-efficient and fast LoRA training of the **Qwen2.5-3B-Instruct** model.

## 1. Setup Environment and Imports
We load our dependencies and identify the compute device capability to determine if bfloat16 is supported.

In [ ]:
import os
import torch
import warnings
from datasets import load_dataset
from transformers import AutoTokenizer
from unsloth import FastLanguageModel, PatchDPOTrainer
from trl import DPOTrainer, DPOConfig

warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8 else torch.float16
print(f'Using device: cuda | Dtype: {compute_dtype}')

## 2. Load Model & Enable PEFT
We load `unsloth/Qwen2.5-3B-Instruct-bnb-4bit` using Unsloth's optimized 4-bit precision to fit within small VRAM budgets, and attach LoRA adapters to all projection layers.

In [ ]:
MODEL_ID = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=1024,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    lora_alpha=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=3407,
)
print('Model and PEFT adapters loaded successfully.')

## 3. Load and Format Dataset (Orca DPO Pairs)
We load the `Intel/orca_dpo_pairs` dataset from Hugging Face, shuffle it, and select a small subset of 250 training examples and 50 validation examples. We map the inputs to a standard DPO format where the user prompt is compiled using the model's native chat template.

In [ ]:
dataset = load_dataset('Intel/orca_dpo_pairs', split='train')
shuffled = dataset.shuffle(seed=42)
train_ds = shuffled.select(range(250))
eval_ds = shuffled.select(range(250, 300))

def format_dpo_example(example):
    messages = [{'role': 'user', 'content': example['question']}]
    prompt_str = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return {
        'prompt': prompt_str,
        'chosen': example['chosen'],
        'rejected': example['rejected']
    }

train_mapped = train_ds.map(format_dpo_example, remove_columns=train_ds.column_names)
eval_mapped = eval_ds.map(format_dpo_example, remove_columns=eval_ds.column_names)
print('Prompt Preview:\n', train_mapped[0]['prompt'])
print('Chosen Preview:\n', train_mapped[0]['chosen'])
print('Rejected Preview:\n', train_mapped[0]['rejected'])

## 4. Run Aligned DPO Fine-Tuning
We run Unsloth's patched version of the TRL `DPOTrainer` (`PatchDPOTrainer()`). We run training for 60 steps with a learning rate of 5e-6, using cosine annealing decay.

In [ ]:
PatchDPOTrainer()

training_args = DPOConfig(
    output_dir='qwen2.5-3b-dpo-output',
    beta=0.1,
    max_length=1024,
    max_prompt_length=512,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    max_steps=60,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    bf16=(compute_dtype == torch.bfloat16),
    fp16=(compute_dtype == torch.float16),
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=20,
    save_steps=20,
    report_to='none',
)

trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=training_args,
    train_dataset=train_mapped,
    eval_dataset=eval_mapped,
    processing_class=tokenizer,
)

model.config.use_cache = False
trainer.train()

model.save_pretrained('qwen2.5-3b-dpo-adapter')
tokenizer.save_pretrained('qwen2.5-3b-dpo-adapter')
print('DPO adapter successfully saved!')

## 5. Evaluation and Inference comparison
We put the model in inference mode and query it with a test prompt to check if it adheres to Aligned Response preferences.

In [ ]:
FastLanguageModel.for_inference(model)

test_question = "Explain why the sky is blue in one concise sentence."
messages = [{'role': 'user', 'content': test_question}]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors='pt').to('cuda')

with torch.no_grad():
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=100,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True).strip()
print('Question:', test_question)
print('\nAligned Response:', response)